In [0]:
-- ╔════════════════════════════════════════════════════════════╗
-- ║  04_Tip_Silver_Cleaned                                    ║
-- ║  Silver_Cleaned Tip Table — Empty DDL Only                ║
-- ╚════════════════════════════════════════════════════════════╝
--
-- Source : yelp_poc_uc.bronze_final.yelp_tip (~908,915 rows)
-- Targets (all in yelp_poc_uc.silver_cleaned):
--   yelp_tip  (no PK; 1 row per tip event)
--
-- ════════════════════════════════════════════════════════════
-- RULES & DESIGN PRINCIPLES
-- ════════════════════════════════════════════════════════════
-- (Complete definitions in 01_Business_Silver_Cleaned)
--
-- R1  DDL-Only / No Population
--     Table is created empty. Data population (type correction)
--     is handled by a separate loading script.
--
-- R2  No Unnecessary Constraint Removal
--     FK/NOT NULL declared inline in CREATE TABLE.
--
-- R4  Referential Integrity via FK
--     FK to yelp_business declared inline. FK to yelp_user is
--     DEFERRED — enabled in 05_User_Silver_Cleaned.
--
-- R5  Semantic Data Types
--     Bronze Type       →  Silver Semantic Type  → Columns Affected
--     ────────────────────────────────────────────────────────────
--     STRING datetime   →  TIMESTAMP             → tip_timestamp
--     BIGINT            →  BIGINT (as-is)        → compliment_count
--
-- R6  No Ingestion Timestamps
--     Lineage tracked via TBLPROPERTIES metadata.
--
-- R7  Self-Documenting DDL
--     Every column carries an inline comment.
--
-- NOTE on Primary Key:
--     No PK is declared. EDA confirmed that the natural
--     composite key (user_id, business_id, tip_timestamp, text)
--     has ~288 duplicate groups. Deduplication is deferred to
--     Silver_Normalized where a surrogate key will be added.
--
-- Execution order: Run AFTER 01_Business (FK).
--   FK to yelp_user DEFERRED — enabled in 05_User.
-- ════════════════════════════════════════════════════════════


-- ── 1. Tip Table (Silver Cleaned) ───────────────────────────
-- Flat table. No nested structures requiring child tables.
-- One row per tip event. No PK due to composite key duplicates.

CREATE OR REPLACE TABLE yelp_poc_uc.silver_cleaned.yelp_tip (
  user_id           STRING    NOT NULL,  -- Tip author's user ID; FK to yelp_user (deferred to 05)
  business_id       STRING    NOT NULL,  -- Tipped business ID; FK to yelp_business
  text              STRING    NOT NULL,  -- Full tip text body
  tip_timestamp     TIMESTAMP NOT NULL,  -- When the tip was posted (source: STRING → TIMESTAMP)
  compliment_count  BIGINT,              -- Number of compliments received for this tip
  CONSTRAINT fk_sc_tip_business
    FOREIGN KEY (business_id)
    REFERENCES yelp_poc_uc.silver_cleaned.yelp_business (business_id)
  -- FK to yelp_user DEFERRED — enabled in 05_User_Silver_Cleaned
  -- CONSTRAINT fk_sc_tip_user
  --   FOREIGN KEY (user_id)
  --   REFERENCES yelp_poc_uc.silver_cleaned.yelp_user (user_id)
)
USING DELTA
COMMENT 'Silver_Cleaned tip table (DDL-only, no data). Type-corrected: date STRING→TIMESTAMP. No PK — composite key has ~288 duplicate groups; dedup deferred to Silver_Normalized. FK to yelp_business; deferred FK to yelp_user. ~908,915 rows when populated.'
TBLPROPERTIES (
  'quality'                = 'silver_cleaned',
  'pipelines.sourceTable'  = 'yelp_poc_uc.bronze_final.yelp_tip',
  'design.populationMode'  = 'ddl_only'
);


-- ── 2. Verification ─────────────────────────────────────────

SELECT 'yelp_tip' AS table_name, 'silver_cleaned' AS layer, COUNT(*) AS total_rows
FROM yelp_poc_uc.silver_cleaned.yelp_tip;